In [1]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [2]:
url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv"

df = pd.read_csv(url)

df.head()

,observed,location_details,county,state,season,title,latitude,longitude,date,number,...,precip_intensity,precip_probability,precip_type,pressure,summary,uv_index,visibility,wind_bearing,wind_speed,location
0,Ed L. was salmon fishing with a companion in P...,East side of Prince William Sound,Valdez-Chitina-Whittier County,Alaska,Fall,NaN,NaN,NaN,NaN,1261.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heh i kinda feel a little dumb that im reporti...,"the road is off us rt 80, i dont know the exit...",Warren County,New Jersey,Fall,NaN,NaN,NaN,NaN,438.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I was on my way to Claremont from Lebanon on R...,Close to Claremont down 120 not far from Kings...,Sullivan County,New Hampshire,Summer,Report 55269: Dawn sighting at Stevens Brook o...,43.41549,-72.33093,2016-06-07,55269.0,...,0.001,0.7,rain,998.87,Mostly cloudy throughout the day.,6.0,9.70,262.0,0.49,POINT(-72.33093000000001 43.415490000000005)
3,I was northeast of Macy Nebraska along the Mis...,Latitude & Longitude : 42.158230 -96.344197,Thurston County,Nebraska,Spring,Report 59757: Possible daylight sighting of a ...,42.15685,-96.34203,2018-05-25,59757.0,...,0.000,0.0,NaN,1008.07,Partly cloudy in the morning.,10.0,8.25,193.0,3.33,POINT(-96.34203000000001 42.15685)
4,"While this incident occurred a long time ago, ...","Ward County, Just outside of a the Minuteman T...",Ward County,North Dakota,Spring,Report 751: Hunter describes described being s...,48.25422,-101.31660,2000-04-21,751.0,...,NaN,NaN,rain,1011.47,Partly cloudy until evening.,6.0,10.00,237.0,11.14,POINT(-101.3166 48.254220000000004)


In [3]:
print(df.shape)
print(df.columns.tolist())
df.info()

(4747, 29)
['observed', 'location_details', 'county', 'state', 'season', 'title', 'latitude', 'longitude', 'date', 'number', 'classification', 'geohash', 'temperature_high', 'temperature_mid', 'temperature_low', 'dew_point', 'humidity', 'cloud_cover', 'moon_phase', 'precip_intensity', 'precip_probability', 'precip_type', 'pressure', 'summary', 'uv_index', 'visibility', 'wind_bearing', 'wind_speed', 'location']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4747 entries, 0 to 4746
Data columns (total 29 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   observed            4711 non-null   object 
 1   location_details    4008 non-null   object 
 2   county              4747 non-null   object 
 3   state               4747 non-null   object 
 4   season              4747 non-null   object 
 5   title               3797 non-null   object 
 6   latitude            3797 non-null   float64
 7   longitude           3797 non-null

In [4]:
df.columns = df.columns.str.strip().str.lower()

df = df.dropna(subset=["state"]).copy()
df["state"] = df["state"].astype(str).str.strip()

state_counts = (
    df.groupby("state")
    .size()
    .reset_index(name="sightings")
    .sort_values("sightings", ascending=False)
)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["year"] = df["date"].dt.year

year_counts = (
    df.dropna(subset=["year"])
    .groupby(["year", "state"])
    .size()
    .reset_index(name="sightings")
)

year_counts["year"] = year_counts["year"].astype(int)

print(state_counts.head())
print(year_counts.head())


         state  sightings
45  Washington        563
4   California        402
8      Florida        292
33        Ohio        276
35      Oregon        241
   year         state  sightings
0  1869    California          1
1  1921  Pennsylvania          1
2  1925     Louisiana          1
3  1930         Texas          1
4  1937       Indiana          1


In [5]:

top_states = state_counts.head(15)

chart1 = alt.Chart(top_states).mark_bar().encode(
    x=alt.X("sightings:Q", title="Number of Sightings"),
    y=alt.Y("state:N", sort="-x", title="State"),
    color=alt.Color(
        "sightings:Q",
        scale=alt.Scale(scheme="blues"),
        title="Sightings"
    ),
    tooltip=[
        alt.Tooltip("state:N", title="State"),
        alt.Tooltip("sightings:Q", title="Sightings")
    ]
).properties(
    title="Top 15 States with Bigfoot Sightings",
    width=600,
    height=400
)

chart1

alt.Chart(...)

In [6]:
states = sorted(year_counts["state"].unique().tolist())

dropdown = alt.binding_select(
    options=["All"] + states,
    name="Select State: "
)

state_select = alt.param(
    name="selected_state",
    value="All",
    bind=dropdown
)

chart2 = alt.Chart(year_counts).mark_line(
    point=True,
    color="steelblue"
).encode(
    x=alt.X("year:O", title="Year"),
    y=alt.Y("sum(sightings):Q", title="Number of Sightings"),
    tooltip=[
        alt.Tooltip("year:O", title="Year"),
        alt.Tooltip("sum(sightings):Q", title="Sightings")
    ]
).add_params(
    state_select
).transform_filter(
    (state_select == "All") |
    (alt.datum.state == state_select)
).properties(
    title="Bigfoot Sightings Over Time",
    width=650,
    height=350
)

chart2

alt.Chart(...)

In [7]:
chart1.save("bigfoot_states.html")
chart2.save("bigfoot_years.html")

In [8]:
from google.colab import files

files.download("bigfoot_states.html")
files.download("bigfoot_years.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>